자연어처리


In [21]:
import pandas as pd
from konlpy.tag import Okt

In [22]:
train_df = pd.read_table("data/ratings_train.txt")
test_df = pd.read_table("data/ratings_test.txt")

#train_df = (variable) test_df: DataFrame
train_df = train_df.fillna(" ")
test_df = test_df.fillna(" ")

In [ ]:
okt = Okt() #Okt는 트위터사에서 만듬, 문법체계는 영미권임
text = "한글 자연어 처리는 재밌다. 이제부터 열심히 해야지 ㅎㅎㅎ"

print(okt.morphs(text)) #morphs는 형태소 단위로 분리
print(okt.morphs(text, stem=True)) 
print(okt.nouns(text)) #명사만 출력 : 한국어는 명사 없고 체언이라 사용함.
print(okt.phrases(text))
print(okt.pos(text))

['한글', '자연어', '처리', '는', '재밌다', '.', '이제', '부터', '열심히', '해야지', 'ㅎㅎㅎ']
['한글', '자연어', '처리', '는', '재밌다', '.', '이제', '부터', '열심히', '하다', 'ㅎㅎㅎ']
['한글', '자연어', '처리', '이제']
['한글', '한글 자연어', '한글 자연어 처리', '이제', '자연어', '처리']
[('한글', 'Noun'), ('자연어', 'Noun'), ('처리', 'Noun'), ('는', 'Josa'), ('재밌다', 'Adjective'), ('.', 'Punctuation'), ('이제', 'Noun'), ('부터', 'Josa'), ('열심히', 'Adverb'), ('해야지', 'Verb'), ('ㅎㅎㅎ', 'KoreanParticle')]


In [30]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vec = TfidfVectorizer(ngram_range=(1,2), min_df=3, max_df=0.9)

아래 임베딩(시간소요됨)을 해야 학습할수있다.

In [32]:
def tw_tokenzier(text):
    tokenzier_ko = okt.morphs(text)
    return tokenzier_ko

from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vec = TfidfVectorizer(tokenizer=tw_tokenzier,
                            token_pattern=None,
                            ngram_range=(1,2),
                            min_df=3,
                            max_df=0.9)
tfidf_vec.fit(train_df["document"])
tfidf_matrix_train = tfidf_vec.transform(train_df["document"])

In [35]:
#학습 아닌것부터알아야한다, 숫자를가지고 평가해야하는데 문자가 들어오면안된다.
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(C=3.5, random_state=42)
lr.fit(tfidf_matrix_train, train_df["label"])

,penalty,'l2'
,dual,False
,tol,0.0001
,C,3.5
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [37]:
#확률을 뽑는다.
#문자넣으면 안된다.
from sklearn.metrics import accuracy_score
tfidf_matrix_test = tfidf_vec.transform(test_df["document"])
y_pred = lr.predict(tfidf_matrix_test)
accuracy_score(test_df["label"], y_pred)

0.86532

In [ ]:
#전처리 덤프뜨기
#웹에서 불러 사용하기
#피클파일이다. 
#메모리의 데이터를 0과1로 바꾼 시리얼라이즈를 한것이다.
import joblib
joblib.dump(lr, "model/lr_v1.pkl")
joblib.dump(tfidf_vec, "model/tfidf_vec_v1.pkl")

['model/tfidf_vec_v1.pkl']

In [48]:
from flask import Flask

app = Flask(__name__)

@app.route("/")
def hello_world():
    return "<p>Hello, World!</p>"

if __name__ == "__main__":
    app.run(debug=True, host="0.0.0.0", port=8000)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8000
 * Running on http://10.125.121.211:8000
Press CTRL+C to quit
 * Restarting with stat


SystemExit: 1

In [49]:
from flask import Flask, render_template

app = Flask(__name__)

@app.route("/")
def hello_world():
    #templates 폴더를 만들고, index.html을 작성하세요
    return render_template("index.html")

if __name__ == "__main__":
    app.run(debug=True, host="0.0.0.0", port=8000)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8000
 * Running on http://10.125.121.211:8000
Press CTRL+C to quit
 * Restarting with stat


SystemExit: 1